# Parametric bootstrap and delta-beta influence for binomial GLMs

GLM.jl fits generalized linear models, but has no parametric bootstrap and no case-deletion influence statistics for them. Two are written out here for binomial GLMs: `parametric` takes the fitted model as true, draws new responses from it and refits, giving intervals for any prediction (Efron 1979); `deltabeta` refits without each case in turn and measures the change in one coefficient (Dalal, Fowlkes and Hoadley 1989). They are tested against the Challenger O-ring analysis of Dalal et al.: its bootstrap interval for the expected damage at 30°F and its delta beta of each of the 23 flights. Both published tests are coarse: an interval "about (1, 6)" and one flight "about four standard deviations" out.

The data is `challenger`, from PollisDatasets: Table 1 of the paper, the O-ring damage of the 23 flights whose rocket motors were recovered, with the temperature of each.

In [ ]:
Pollis.packages("GLM", "Distributions", "PollisDatasets", "Plots")

## Set-up

`replicate` prints each value in full, rounded to the published precision, beside the published value, and how many decimals they share. `flights` holds the temperature and the share of the six field-joint primary O-rings of each flight that suffered erosion or blowby; `six` gives the binomial n of each flight as its weight.

In [ ]:
using GLM, Distributions, PollisDatasets, Plots, Printf, Random, Statistics

"""
	replicate(title, rows)

Print each Julia value in full, rounded to the published precision and beside the published value, with the number of decimals they share.
Each row is `(label, value, published)`, with `published` written as in the source: its decimals give the published precision.
"""
function replicate(title, rows)
	@printf("\n%s\n%-30s%16s%14s%14s\n", title, "", "Julia", "rounded", "published")
	for (label, value, published) in rows
		p = parse(Float64, published)
		d = occursin('.', published) ? length(split(published, '.')[end]) : 0  # decimals as published
		k = findlast(j -> round(value, digits=j) == round(p, digits=j), 0:d)  # position in 0:d
		@printf("%-30s%16.8f%14s%14s  %s\n", label, value, Printf.format(Printf.Format("%.$(d)f"), value), published, k === nothing ? "DIFFERENT" : "same to $(k - 1) decimals")
	end
end

ch = dataset("challenger");  # one row per flight, Table 1 (p. 949)
flights = (temperature = Float64.(ch.temperature), distressed = ch.n_failures ./ 6);  # share of the six field-joint O-rings damaged
six = fill(6.0, 23);  # six field joints per flight: the binomial n

## The method

### Parametric bootstrap

For a binomial GLM with n trials per case, written as a share y with weights n, the parametric bootstrap takes the fitted probabilities p̂ as true, draws y* ~ Binomial(n, p̂)/n for every case, refits the model to y*, and repeats B times (Efron 1979). The spread of any refitted quantity, a coefficient or a prediction, estimates its sampling distribution, without asymptotic formulas. A sample whose fit fails, as one with all successes or none can, is skipped and counted.

In [ ]:
"""
	parametric(m, f, data, response, n, B; rng=Random.default_rng())

`B` refits of the binomial GLM formula `f` to responses drawn from the fitted model `m`: each case's share `response` in `data` is redrawn
as Binomial(n, p̂)/n, with `n` the trials of each case. Returns the refitted models; a sample whose fit fails is skipped.
"""
function parametric(m, f, data, response, n, B; rng=Random.default_rng())
	p = predict(m)
	fits = []
	for _ in 1:B
		y = rand.(rng, Binomial.(round.(Int, n), p)) ./ n
		try
			push!(fits, glm(f, merge(data, NamedTuple{(response,)}((y,))), Binomial(); wts=n))
		catch
		end
	end
	fits
end

### Delta beta

Case i's influence on coefficient j is the change in that coefficient when the case is dropped and the model refitted: δᵢ = (β̂ⱼ − β̂ⱼ₍₋ᵢ₎) / S, scaled by S, the standard deviation of the n refitted coefficients (Dalal et al. 1989, p. 950). Unlike the one-step approximations of Pregibon (1981), it refits exactly, which matters when a few cases carry the fit.

In [ ]:
"""
	deltabeta(f, data, n, j)

The change in coefficient `j` of the binomial GLM formula `f` when each case of `data`, with trials `n`, is dropped in turn, divided by the
standard deviation of the refitted coefficients. Returns the scaled changes and the refitted coefficients.
"""
function deltabeta(f, data, n, j)
	all = coef(glm(f, data, Binomial(); wts=n))[j]
	N = length(n)
	loo = [coef(glm(f, map(column -> column[setdiff(1:N, i)], data), Binomial(); wts=n[setdiff(1:N, i)]))[j] for i in 1:N]
	(all .- loo) ./ std(loo), loo
end

## Tests

The model is Dalal et al.'s Model (3.2): the share of damaged field-joint O-rings, binomial with n = 6, logistic in the temperature (p. 948). Its fit is checked first against the paper, since both tests rest on it.

In [ ]:
f = @formula(distressed ~ temperature)
model = glm(f, flights, Binomial(); wts=six)
replicate("Model (3.2) (p. 948)", collect(zip(["alpha", "beta", "se alpha", "se beta", "G2, 21 df"],
	[coef(model); stderror(model); deviance(model)], ["5.085", "-.1156", "3.052", ".047", "18.086"])))

### The bootstrap interval at 30°F

The paper draws bootstrap samples from Model (3.2), refits, and takes the 5% and 95% points of the expected number of damaged O-rings, 6 p(t), at each temperature: a 90% interval, wide below 65°F where there are few flights. At 30°F it is "about (1, 6)" (p. 950, Fig. 5). A bootstrap is random, so the last digits change with the seed; 1,000 samples are drawn here.

In [ ]:
grid = 30:85
new(t) = (temperature = Float64.(collect(t)),)
boot = parametric(model, f, flights, :distressed, six, 1000; rng=Xoshiro(1989))
curves = reduce(hcat, [6predict(b, new(grid)) for b in boot])
lower, upper = quantile.(eachrow(curves), 0.05), quantile.(eachrow(curves), 0.95)
@printf("%d of 1000 bootstrap fits succeeded\n", length(boot))
replicate("90% bootstrap interval at 30°F (p. 950)", [("5%", lower[1], "1"), ("95%", upper[1], "6")])
fitted = 6predict(model, new(grid))
plot(grid, fitted, ribbon=(fitted .- lower, upper .- fitted), fillalpha=0.25, label="expected, with 90% bootstrap interval",
	xlabel="joint temperature, °F", ylabel="expected damaged O-rings", ylims=(0, 6), size=(560, 330))
scatter!(flights.temperature, ch.n_failures, label="flights")

### The delta beta of each flight

The paper's delta beta of the temperature coefficient picks out one flight by about four standard deviations: the 21st, 61-A, with two damaged O-rings at 75°F. The next largest are flights 2, 11 and 14, all with damage (p. 950, Fig. 7).

In [ ]:
delta, loo = deltabeta(f, flights, six, 2)
replicate("Delta beta of the temperature coefficient (p. 950)", [("flight 21 ($(ch.flight[21]))", delta[21], "4")])
@printf("the three next largest: flights %s (published: 2, 11 and 14)\n", join(sort(sortperm(abs.(delta), rev=true)[2:4]), ", "))
scatter(1:23, delta, xlabel="flight, in launch order", ylabel="delta beta", label="", series_annotations=text.(ch.flight, 7, :bottom), size=(560, 330))

## Example

How much does flight 61-A drive the cold-weather prediction? The bootstrap interval at 30°F with that flight dropped, beside the interval with all 23: without its damage at a warm temperature, the model expects nearly all six O-rings to be damaged, and the interval narrows to the top of the scale.

In [ ]:
keep = setdiff(1:23, 21)
without = map(column -> column[keep], flights)
m21 = glm(f, without, Binomial(); wts=six[keep])
b21 = parametric(m21, f, without, :distressed, six[keep], 1000; rng=Xoshiro(61))
e21 = [6predict(b, new([30]))[1] for b in b21]
@printf("expected damaged O-rings at 30°F, all flights: %.2f, 90%% interval (%.2f, %.2f)\n", fitted[1], lower[1], upper[1])
@printf("without flight 61-A:                          %.2f, 90%% interval (%.2f, %.2f)\n", 6predict(m21, new([30]))[1], quantile(e21, 0.05), quantile(e21, 0.95))

## What differs

Every value agrees to the published precision: the coefficients, standard errors and G² of Model (3.2); the bootstrap interval at 30°F, (1.27, 5.95) for "about (1, 6)"; the delta beta of flight 61-A, 4.22 for "about four"; and the next three flights, 2, 11 and 14.

The tests are coarse, because the paper's are: it prints the interval and the delta beta to no decimals and shows the rest in figures. They confirm that the methods do what the paper did, not their last digits. The bootstrap is random, and its interval moves in the second decimal with the seed.

The paper also bootstraps a model of primary erosion with temperature and pressure (p. 953). About a third of its samples have no maximum likelihood estimate: one coefficient runs off to infinity, and the paper does not say what its program did with them. Those intervals are not used as tests here; the Logistic regression workflow shows them.

## References

- Dalal, S. R., Fowlkes, E. B. and Hoadley, B. (1989). Risk analysis of the space shuttle: pre-Challenger prediction of failure. *Journal of the American Statistical Association* 84(408), 945-957. doi:10.1080/01621459.1989.10478858
- Efron, B. (1979). Bootstrap methods: another look at the jackknife. *The Annals of Statistics* 7(1), 1-26.
- Pregibon, D. (1981). Logistic regression diagnostics. *The Annals of Statistics* 9(4), 705-724.
- Presidential Commission on the Space Shuttle Challenger Accident (1986). *Report to the President*, Vol. 1, pp. 129-131: the data.